# LeetCode 436 — Find Right Interval bằng Python

Link đề: https://leetcode.com/problems/find-right-interval/ · Độ khó: **Medium** · Chủ đề: Array, Binary Search, Sorting

Skeleton LeetCode:

```python
class Solution:
    def findRightInterval(self, intervals: List[List[int]]) -> List[int]:
        ...
```

Ghi chú: đề và các ví dụ được viết lại theo hiểu biết về đề gốc; mọi đáp án trong notebook đã được kiểm chứng bằng code (so với cách vét cạn), nhưng nên đối chiếu lại đề gốc nếu muốn khớp từng chữ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: với mỗi đoạn `i`, tìm đoạn `j` có **điểm bắt đầu nhỏ nhất** trong số những đoạn có `start_j >= end_i` (gọi là **right interval**); không có thì ghi `-1`. Lưu ý `j` được phép bằng `i`.
2. Nắm các công cụ Python: `enumerate`, `sorted` với `tuple`, `bisect_left`, `dict`, `heapq` (hàng đợi ưu tiên), và hai con trỏ chạy trên hai danh sách đã sắp.
3. Đi từ **quét toàn bộ cho từng đoạn `O(n²)`** → **sắp xếp điểm bắt đầu + tìm nhị phân** → **sắp cả hai phía rồi hai con trỏ** → **min-heap loại dần các start quá nhỏ**.
4. Hiểu mấu chốt: bài là "**với mỗi giá trị `end`, tìm phần tử nhỏ nhất `>= end` trong tập các `start`**" (bài toán *lower bound*), và ta phải **nhớ chỉ số gốc** trước khi sắp xếp.
5. Tránh các lỗi kinh điển: sắp xếp mất chỉ số gốc, dùng `bisect_right` thay `bisect_left` (bỏ sót `start == end`), quên trường hợp `j == i` và quên trả `-1` khi đi quá cuối mảng.

## 2. Đề bài & ví dụ

Cho mảng `intervals` với `intervals[i] = [start_i, end_i]` và **mọi `start_i` đều khác nhau**.

**Right interval** của đoạn `i` là đoạn `j` sao cho `start_j >= end_i` và `start_j` **nhỏ nhất** có thể. Chú ý `i` có thể bằng `j`.

Hãy trả về mảng `ans`, trong đó `ans[i]` là **chỉ số** của right interval của đoạn `i`, hoặc `-1` nếu không tồn tại.

**Ví dụ 1**

- Input: `intervals = [[1,2]]`
- Output: `[-1]`
- Giải thích: chỉ có một đoạn, và `start = 1 < end = 2` nên không có đoạn nào thoả.

**Ví dụ 2**

- Input: `intervals = [[3,4],[2,3],[1,2]]`
- Output: `[-1,0,1]`
- Giải thích: đoạn `[3,4]` không có right interval. Đoạn `[2,3]` cần `start >= 3`: có `start_0 = 3` nên đáp án `0`. Đoạn `[1,2]` cần `start >= 2`: có `start_1 = 2` nên đáp án `1`.

**Ví dụ 3**

- Input: `intervals = [[1,4],[2,3],[3,4]]`
- Output: `[-1,2,-1]`
- Giải thích: `[1,4]` và `[3,4]` cần `start >= 4`, không có. `[2,3]` cần `start >= 3`: có `start_2 = 3`, đáp án `2`.

**Ràng buộc:**

- `1 <= intervals.length <= 2 * 10^4`
- `intervals[i].length == 2`
- `-10^6 <= start_i <= end_i <= 10^6`
- Điểm bắt đầu của mỗi đoạn là **duy nhất**.

In [1]:
from typing import List
import bisect
import heapq
import random

examples = [
    ([[1, 2]], [-1]),
    ([[3, 4], [2, 3], [1, 2]], [-1, 0, 1]),
    ([[1, 4], [2, 3], [3, 4]], [-1, 2, -1]),
]
for intervals, expected in examples:
    print(intervals, "-> kỳ vọng", expected)

[[1, 2]] -> kỳ vọng [-1]
[[3, 4], [2, 3], [1, 2]] -> kỳ vọng [-1, 0, 1]
[[1, 4], [2, 3], [3, 4]] -> kỳ vọng [-1, 2, -1]


## 3. Kiến thức nền

### 3.1. `enumerate` và giải nén `(i, (l, r))`

Ta luôn cần **chỉ số gốc** `i` của mỗi đoạn vì đáp án là chỉ số. `enumerate(intervals)` cho từng cặp `(i, đoạn)`; có thể giải nén luôn đoạn: `for i, (l, r) in enumerate(intervals)`.

In [2]:
intervals = [[3, 4], [2, 3], [1, 2]]
for i, (l, r) in enumerate(intervals):
    print(f"đoạn {i}: start={l}, end={r}")

đoạn 0: start=3, end=4
đoạn 1: start=2, end=3
đoạn 2: start=1, end=2


### 3.2. Sắp xếp list các tuple: so sánh từng phần tử một

`sorted([(st, i), ...])` sắp theo `st` trước; nếu `st` bằng nhau mới xét `i`. Vì `start` là duy nhất nên chỉ cần `st`. Cách đóng gói `(start, index)` giúp **mang chỉ số gốc theo** sau khi sắp.

In [3]:
arr = sorted((l, i) for i, (l, _) in enumerate(intervals))
print(arr)          # [(1, 2), (2, 1), (3, 0)] : (start, chỉ số gốc)

[(1, 2), (2, 1), (3, 0)]


### 3.3. `bisect_left`: "lower bound" — vị trí đầu tiên có giá trị `>= x`

Trên danh sách đã sắp tăng, `bisect_left(a, x)` trả về chỉ số **nhỏ nhất** `k` sao cho `a[k] >= x`; nếu mọi phần tử đều `< x` thì trả `len(a)`. Đó chính xác là thứ ta cần: "start nhỏ nhất mà `>= end`". Còn `bisect_right` trả về vị trí đầu tiên có `a[k] > x` (**lớn hơn hẳn**), nên sẽ **bỏ sót** phần tử bằng `x`.

In [4]:
starts = [1, 2, 3, 7]
for x in [0, 2, 3, 4, 7, 8]:
    k = bisect.bisect_left(starts, x)
    print(f"x={x}: bisect_left={k}", "->", starts[k] if k < len(starts) else "không có",
          f"| bisect_right={bisect.bisect_right(starts, x)}")

x=0: bisect_left=0 -> 1 | bisect_right=0
x=2: bisect_left=1 -> 2 | bisect_right=2
x=3: bisect_left=2 -> 3 | bisect_right=3
x=4: bisect_left=3 -> 7 | bisect_right=3
x=7: bisect_left=3 -> 7 | bisect_right=4
x=8: bisect_left=4 -> không có | bisect_right=4


### 3.4. `dict`: ánh xạ `start -> chỉ số gốc`

Vì mọi `start` là duy nhất, ta có thể dùng `dict` `idx_of[start] = i` để sau khi tìm được một giá trị `start` ta tra ra chỉ số gốc ngay trong `O(1)`.

In [5]:
idx_of = {l: i for i, (l, _) in enumerate(intervals)}
print(idx_of)             # {3: 0, 2: 1, 1: 2}
print(idx_of[2])          # 1

{3: 0, 2: 1, 1: 2}
1


### 3.5. `heapq`: min-heap

`heapq` giữ một list sao cho phần tử **nhỏ nhất luôn ở `heap[0]`**. `heapify(list)` biến list thành heap trong `O(n)`; `heappop(heap)` lấy phần tử nhỏ nhất ra trong `O(log n)`. Dùng tuple `(start, index)` để cũng mang theo chỉ số.

In [6]:
h = [(7, 0), (2, 1), (5, 2), (1, 3)]
heapq.heapify(h)
print(h[0])                        # (1, 3) nhỏ nhất
print(heapq.heappop(h), heapq.heappop(h))   # (1, 3) (2, 1)
print(h[0])                        # (5, 2)

(1, 3)
(1, 3) (2, 1)
(5, 2)


### 3.6. Hai con trỏ trên hai danh sách đã sắp

Nếu danh sách câu hỏi `q` cũng được sắp tăng, thì "vị trí lower bound" của `q` **chỉ có thể dịch sang phải** khi `q` tăng. Thay vì tìm nhị phân từng lần, ta giữ một con trỏ `p` chỉ ngày một đi tới. Mỗi con trỏ đi tối đa `n` bước nên phần quét là `O(n)`.

In [7]:
a = [1, 2, 3, 7]          # đã sắp
qs = [0, 2, 4, 8]         # đã sắp
p = 0
for q in qs:
    while p < len(a) and a[p] < q:
        p += 1
    print(f"q={q}: con trỏ p={p}", "->", a[p] if p < len(a) else "hết")

q=0: con trỏ p=0 -> 1
q=2: con trỏ p=1 -> 2
q=4: con trỏ p=3 -> 7
q=8: con trỏ p=4 -> hết


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Với `intervals = [[3,4],[2,3],[1,2]]` ta liệt kê `start` và `end` của từng đoạn:

| i | start | end | cần `start >=` | các start thoả | start nhỏ nhất | đáp án |
|---|---|---|---|---|---|---|
| 0 | 3 | 4 | 4 | (không) | - | `-1` |
| 1 | 2 | 3 | 3 | 3 (đoạn 0) | 3 | `0` |
| 2 | 1 | 2 | 2 | 2 (đoạn 1), 3 (đoạn 0) | 2 | `1` |

Mỗi đoạn là một "câu hỏi" độc lập: **"trong tập các `start`, giá trị nhỏ nhất không bé hơn `end_i` là gì, và nó thuộc đoạn nào?"**

In [8]:
intervals = [[3, 4], [2, 3], [1, 2]]
for i, (l, r) in enumerate(intervals):
    ok = [(s, j) for j, (s, _) in enumerate(intervals) if s >= r]
    print(f"i={i}: cần start >= {r}; các ứng viên (start, j) = {ok}; chọn", min(ok) if ok else "-1")

i=0: cần start >= 4; các ứng viên (start, j) = []; chọn -1
i=1: cần start >= 3; các ứng viên (start, j) = [(3, 0)]; chọn (3, 0)
i=2: cần start >= 2; các ứng viên (start, j) = [(3, 0), (2, 1)]; chọn (2, 1)


## 5. Cách 1 — Brute Force / baseline: quét toàn bộ cho mỗi đoạn

**Ý tưởng.** Với mỗi `i`, duyệt mọi `j`; giữ lại `j` có `start_j >= end_i` và `start_j` nhỏ nhất. Đúng từng chữ theo đề.

In [9]:
def find_right_bruteforce(intervals):
    n = len(intervals)
    ans = []
    for i in range(n):
        best_start = None
        best_j = -1
        for j in range(n):
            s = intervals[j][0]
            if s >= intervals[i][1] and (best_start is None or s < best_start):
                best_start = s
                best_j = j
        ans.append(best_j)
    return ans


for iv, expected in examples:
    print(find_right_bruteforce(iv), "(kỳ vọng", expected, ")")

[-1] (kỳ vọng [-1] )
[-1, 0, 1] (kỳ vọng [-1, 0, 1] )
[-1, 2, -1] (kỳ vọng [-1, 2, -1] )


**Độ phức tạp (Cách 1).** Hai vòng lồng ⇒ **Time `O(n²)`**, **Space `O(1)`** (không kể mảng đáp án). Với `n = 2·10^4` là `4·10^8` thao tác Python — quá chậm.

**Nút thắt:** với mỗi `end` ta lại quét **toàn bộ** `start`. Nếu các `start` đã **sắp tăng**, ta có thể "nhảy" tới vị trí cần tìm bằng tìm kiếm nhị phân.

## 6. Ý tưởng thuật toán / bước đột phá

**Sắp xếp `start` một lần, rồi mỗi câu hỏi chỉ tốn `O(log n)`.**

1. Tạo `arr = sorted((start, index))`.
2. Với mỗi đoạn `i`, `k = bisect_left(arr, (end_i, -inf))` (hoặc tìm trên danh sách riêng chỉ chứa `start`) là vị trí `start` nhỏ nhất `>= end_i`.
3. Nếu `k < n` thì đáp án là chỉ số gốc `arr[k][1]`, ngược lại `-1`.

Vì sao có thể "nhìn ngay" phần tử nhỏ nhất thoả điều kiện? Vì trong danh sách sắp tăng, tập các phần tử `>= x` là **một đoạn liên tục ở cuối**, và phần tử **đầu tiên** của đoạn đó chính là nhỏ nhất.

In [10]:
intervals = [[3, 4], [2, 3], [1, 2]]
starts_sorted = sorted((l, i) for i, (l, _) in enumerate(intervals))
only_starts = [s for s, _ in starts_sorted]
print("starts sắp:", starts_sorted)
for i, (l, r) in enumerate(intervals):
    k = bisect.bisect_left(only_starts, r)
    print(f"i={i}, end={r}: k={k}", "->", starts_sorted[k][1] if k < len(intervals) else -1)

starts sắp: [(1, 2), (2, 1), (3, 0)]
i=0, end=4: k=3 -> -1
i=1, end=3: k=2 -> 0
i=2, end=2: k=1 -> 1


## 7. Các cách tiếp theo

### Cách 2 — Sắp `(start, index)` + `bisect_left` (`O(n log n)`)

**Giải thích từng đoạn.**

1. `arr = sorted((st, i) for i, (st, _) in enumerate(intervals))`: danh sách start kèm chỉ số gốc.
2. `starts = [st for st, _ in arr]`: danh sách chỉ chứa start (để dùng `bisect` với số, không phải tuple).
3. Với mỗi `(i, (_, ed))`: `k = bisect_left(starts, ed)`; nếu `k < n` thì `ans[i] = arr[k][1]`.

In [11]:
def find_right_bisect(intervals):
    n = len(intervals)
    arr = sorted((st, i) for i, (st, _) in enumerate(intervals))
    starts = [st for st, _ in arr]
    ans = [-1] * n
    for i, (_, ed) in enumerate(intervals):
        k = bisect.bisect_left(starts, ed)
        if k < n:
            ans[i] = arr[k][1]
    return ans


for iv, expected in examples:
    print(find_right_bisect(iv), "(kỳ vọng", expected, ")")

[-1] (kỳ vọng [-1] )
[-1, 0, 1] (kỳ vọng [-1, 0, 1] )
[-1, 2, -1] (kỳ vọng [-1, 2, -1] )


In [12]:
# Code nộp LeetCode cho Cách 2
class SolutionBisect:
    def findRightInterval(self, intervals: List[List[int]]) -> List[int]:
        n = len(intervals)
        ans = [-1] * n
        arr = sorted((st, i) for i, (st, _) in enumerate(intervals))
        starts = [st for st, _ in arr]
        for i, (_, ed) in enumerate(intervals):
            j = bisect.bisect_left(starts, ed)
            if j < n:
                ans[i] = arr[j][1]
        return ans


print(SolutionBisect().findRightInterval([[3, 4], [2, 3], [1, 2]]))      # [-1, 0, 1]

[-1, 0, 1]


**Độ phức tạp Cách 2:** sắp xếp `O(n log n)` + `n` lần `bisect` mỗi lần `O(log n)` ⇒ **Time `O(n log n)`**, **Space `O(n)`** cho `arr` và `starts`.

### Cách 3 — Hai con trỏ: sắp cả `start` lẫn `end` (`O(n log n)`)

**Giải thích từng đoạn.**

1. `by_start = sorted((st, i))`: các start tăng dần kèm chỉ số.
2. `by_end = sorted(range(n), key=lambda i: intervals[i][1])`: các **chỉ số đoạn** theo `end` tăng dần.
3. Con trỏ `p` chạy trên `by_start`. Với `i` lần lượt theo `end` tăng: đẩy `p` tới khi `by_start[p][0] >= end_i`. Vì `end` tăng nên `p` **không bao giờ phải lùi**.
4. `ans[i] = by_start[p][1]` nếu `p < n`, ngược lại `-1`.

Khác Cách 2 ở chỗ: không dùng tìm nhị phân, mà tận dụng tính **đơn điệu** của `end` đã sắp.

In [13]:
def find_right_two_pointers(intervals):
    n = len(intervals)
    by_start = sorted((st, i) for i, (st, _) in enumerate(intervals))
    by_end = sorted(range(n), key=lambda i: intervals[i][1])
    ans = [-1] * n
    p = 0
    for i in by_end:
        ed = intervals[i][1]
        while p < n and by_start[p][0] < ed:
            p += 1
        if p < n:
            ans[i] = by_start[p][1]
    return ans


for iv, expected in examples:
    print(find_right_two_pointers(iv), "(kỳ vọng", expected, ")")

[-1] (kỳ vọng [-1] )
[-1, 0, 1] (kỳ vọng [-1, 0, 1] )
[-1, 2, -1] (kỳ vọng [-1, 2, -1] )


In [14]:
# Code nộp LeetCode cho Cách 3
class SolutionTwoPointers:
    def findRightInterval(self, intervals: List[List[int]]) -> List[int]:
        n = len(intervals)
        by_start = sorted((st, i) for i, (st, _) in enumerate(intervals))
        by_end = sorted(range(n), key=lambda i: intervals[i][1])
        ans = [-1] * n
        p = 0
        for i in by_end:
            ed = intervals[i][1]
            while p < n and by_start[p][0] < ed:
                p += 1
            if p < n:
                ans[i] = by_start[p][1]
        return ans


print(SolutionTwoPointers().findRightInterval([[1, 4], [2, 3], [3, 4]]))     # [-1, 2, -1]

[-1, 2, -1]


**Độ phức tạp Cách 3:** hai lần sắp `O(n log n)`; con trỏ `p` chỉ tăng nên phần quét tổng `O(n)` ⇒ **Time `O(n log n)`**, **Space `O(n)`**.

### Cách 4 — Min-heap các start, loại dần những start đã quá nhỏ

**Giải thích từng đoạn.**

1. `heap = [(st, i) ...]` rồi `heapify` (`O(n)`): phần tử nhỏ nhất là start bé nhất còn lại.
2. Duyệt các đoạn theo `end` **tăng dần**. Với `end_i`: `while heap and heap[0][0] < end_i: heappop(heap)`. Những start này **nhỏ hơn `end_i`** nên cũng nhỏ hơn mọi `end` về sau ⇒ vứt đi mãi mãi (không bao giờ cần lại).
3. Nếu heap còn phần tử thì `heap[0][1]` là chỉ số đáp án (start nhỏ nhất `>= end_i`), ngược lại `-1`. **Không pop** phần tử này vì đoạn kế tiếp có thể cũng cần nó.

Ý tưởng là xử lý **offline**: nhận trước mọi câu hỏi, sắp lại để "dọn dẹp dần" cấu trúc dữ liệu.

In [15]:
def find_right_heap(intervals):
    n = len(intervals)
    heap = [(st, i) for i, (st, _) in enumerate(intervals)]
    heapq.heapify(heap)
    ans = [-1] * n
    for i in sorted(range(n), key=lambda i: intervals[i][1]):
        ed = intervals[i][1]
        while heap and heap[0][0] < ed:
            heapq.heappop(heap)
        if heap:
            ans[i] = heap[0][1]
    return ans


for iv, expected in examples:
    print(find_right_heap(iv), "(kỳ vọng", expected, ")")

[-1] (kỳ vọng [-1] )
[-1, 0, 1] (kỳ vọng [-1, 0, 1] )
[-1, 2, -1] (kỳ vọng [-1, 2, -1] )


In [16]:
# Code nộp LeetCode cho Cách 4
class SolutionHeap:
    def findRightInterval(self, intervals: List[List[int]]) -> List[int]:
        n = len(intervals)
        heap = [(st, i) for i, (st, _) in enumerate(intervals)]
        heapq.heapify(heap)
        ans = [-1] * n
        for i in sorted(range(n), key=lambda i: intervals[i][1]):
            ed = intervals[i][1]
            while heap and heap[0][0] < ed:
                heapq.heappop(heap)
            if heap:
                ans[i] = heap[0][1]
        return ans


print(SolutionHeap().findRightInterval([[3, 4], [2, 3], [1, 2]]))     # [-1, 0, 1]

[-1, 0, 1]


**Độ phức tạp Cách 4:** `heapify` `O(n)`, sắp theo `end` `O(n log n)`, mỗi phần tử bị `pop` tối đa một lần (`O(log n)` mỗi lần) ⇒ **Time `O(n log n)`**, **Space `O(n)`**.

## 8. TRACE từng bước

Ta trace Cách 2 (`bisect_left`) và Cách 3 (hai con trỏ) trên `intervals = [[1,4],[2,3],[3,4]]`, rồi Cách 4 (heap) trên `[[3,4],[2,3],[1,2]]`.

In [17]:
def trace_bisect(intervals):
    n = len(intervals)
    arr = sorted((st, i) for i, (st, _) in enumerate(intervals))
    starts = [st for st, _ in arr]
    print("arr (start, chỉ số gốc):", arr)
    print("starts:", starts)
    ans = [-1] * n
    for i, (_, ed) in enumerate(intervals):
        k = bisect.bisect_left(starts, ed)
        if k < n:
            ans[i] = arr[k][1]
        print(f"i={i}: end={ed} -> bisect_left={k} ->", f"arr[{k}]={arr[k]}" if k < n else "vượt cuối", f"=> ans[{i}]={ans[i]}")
    return ans


assert trace_bisect([[1, 4], [2, 3], [3, 4]]) == [-1, 2, -1]

arr (start, chỉ số gốc): [(1, 0), (2, 1), (3, 2)]
starts: [1, 2, 3]
i=0: end=4 -> bisect_left=3 -> vượt cuối => ans[0]=-1
i=1: end=3 -> bisect_left=2 -> arr[2]=(3, 2) => ans[1]=2
i=2: end=4 -> bisect_left=3 -> vượt cuối => ans[2]=-1


In [18]:
def trace_two_pointers(intervals):
    n = len(intervals)
    by_start = sorted((st, i) for i, (st, _) in enumerate(intervals))
    by_end = sorted(range(n), key=lambda i: intervals[i][1])
    print("by_start:", by_start)
    print("by_end (chỉ số đoạn theo end tăng):", by_end)
    ans = [-1] * n
    p = 0
    for i in by_end:
        ed = intervals[i][1]
        while p < n and by_start[p][0] < ed:
            print(f"   end={ed}: start {by_start[p][0]} < {ed} -> p tăng")
            p += 1
        ans[i] = by_start[p][1] if p < n else -1
        print(f"xử lý đoạn {i} (end={ed}): p={p} => ans[{i}]={ans[i]}")
    return ans


assert trace_two_pointers([[1, 4], [2, 3], [3, 4]]) == [-1, 2, -1]

by_start: [(1, 0), (2, 1), (3, 2)]
by_end (chỉ số đoạn theo end tăng): [1, 0, 2]
   end=3: start 1 < 3 -> p tăng
   end=3: start 2 < 3 -> p tăng
xử lý đoạn 1 (end=3): p=2 => ans[1]=2
   end=4: start 3 < 4 -> p tăng
xử lý đoạn 0 (end=4): p=3 => ans[0]=-1
xử lý đoạn 2 (end=4): p=3 => ans[2]=-1


In [19]:
def trace_heap(intervals):
    n = len(intervals)
    heap = [(st, i) for i, (st, _) in enumerate(intervals)]
    heapq.heapify(heap)
    print("heap ban đầu:", sorted(heap))
    ans = [-1] * n
    for i in sorted(range(n), key=lambda i: intervals[i][1]):
        ed = intervals[i][1]
        while heap and heap[0][0] < ed:
            print(f"   end={ed}: bỏ start {heapq.heappop(heap)}")
        ans[i] = heap[0][1] if heap else -1
        print(f"đoạn {i} (end={ed}): đỉnh heap = {heap[0] if heap else None} => ans[{i}]={ans[i]}")
    return ans


assert trace_heap([[3, 4], [2, 3], [1, 2]]) == [-1, 0, 1]

heap ban đầu: [(1, 2), (2, 1), (3, 0)]
   end=2: bỏ start (1, 2)
đoạn 2 (end=2): đỉnh heap = (2, 1) => ans[2]=1
   end=3: bỏ start (2, 1)
đoạn 1 (end=3): đỉnh heap = (3, 0) => ans[1]=0
   end=4: bỏ start (3, 0)
đoạn 0 (end=4): đỉnh heap = None => ans[0]=-1


Bảng trace Cách 2 với `intervals = [[1,4],[2,3],[3,4]]` (`arr = [(1,0),(2,1),(3,2)]`, `starts = [1,2,3]`):

| i | `end_i` | `bisect_left(starts, end_i)` | `arr[k]` | `ans[i]` |
|---|---|---|---|---|
| 0 | 4 | 3 | vượt cuối | `-1` |
| 1 | 3 | 2 | `(3, 2)` | `2` |
| 2 | 4 | 3 | vượt cuối | `-1` |

Kết quả `[-1, 2, -1]`, khớp ví dụ 3.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 2 (sắp `(start, index)` + `bisect_left`): ngắn, dễ nhớ, `O(n log n)`.

In [20]:
class Solution:
    def findRightInterval(self, intervals: List[List[int]]) -> List[int]:
        n = len(intervals)
        ans = [-1] * n
        arr = sorted((st, i) for i, (st, _) in enumerate(intervals))     # (start, chỉ số gốc)
        starts = [st for st, _ in arr]                                  # chỉ các start, để bisect
        for i, (_, ed) in enumerate(intervals):
            j = bisect.bisect_left(starts, ed)                          # start nhỏ nhất >= ed
            if j < n:
                ans[i] = arr[j][1]
        return ans


sol = Solution()
for iv, expected in examples:
    print(iv, "->", sol.findRightInterval(iv))

[[1, 2]] -> [-1]
[[3, 4], [2, 3], [1, 2]] -> [-1, 0, 1]
[[1, 4], [2, 3], [3, 4]] -> [-1, 2, -1]


## 10. Vì sao thuật toán đúng

- **Mang theo chỉ số gốc:** đáp án là chỉ số trong mảng ban đầu, nên cặp `(start, index)` được sắp cùng nhau; sau sắp xếp ta vẫn đọc được `index`.
- **Lower bound:** trong `starts` đã sắp tăng, các phần tử `>= end_i` tạo thành một **hậu tố liên tục**. `bisect_left` trả vị trí đầu tiên của hậu tố đó — phần tử nhỏ nhất `>= end_i`. Đó đúng là điều kiện "start nhỏ nhất và `>= end_i`".
- **`>=` chứ không phải `>`:** đề dùng `start_j >= end_i`, nên `bisect_left` (không phải `bisect_right`).
- **`j == i` được phép:** vì `start_i <= end_i`, chỉ khi `start_i == end_i` thì chính đoạn `i` thoả điều kiện; thuật toán không cấm trường hợp này.
- **`-1`:** khi `k == n` tức không có start nào `>= end_i`.
- **Start duy nhất:** nên "start nhỏ nhất" xác định duy nhất đoạn `j`, không cần phá hoà.
- **Cách 3 và Cách 4:** dựa trên tính đơn điệu — nếu `end_a <= end_b` thì vị trí lower bound của `a` không vượt quá của `b`; con trỏ/heap chỉ cần đi tới, không phải lùi.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(intervals)`.

| Cách | Time | Space |
|---|---|---|
| 1. Quét toàn bộ | `O(n²)` | `O(1)` |
| 2. Sắp + `bisect_left` | `O(n log n)` | `O(n)` |
| 3. Hai con trỏ | `O(n log n)` | `O(n)` |
| 4. Min-heap | `O(n log n)` | `O(n)` |

Với `n = 2·10^4`: `n log n ≈ 3·10^5` thao tác (tức thì), `n² = 4·10^8` (chậm).

## 12. Bộ test

Ta gom các cách vào `impls` và so với kỳ vọng. Mọi hàm đều **không sửa** mảng đầu vào nên có thể truyền trực tiếp.

In [21]:
impls = {
    "bruteforce": find_right_bruteforce,
    "bisect": find_right_bisect,
    "two_pointers": find_right_two_pointers,
    "heap": find_right_heap,
    "leetcode": lambda iv: Solution().findRightInterval(iv),
    "submit_bisect": lambda iv: SolutionBisect().findRightInterval(iv),
    "submit_tp": lambda iv: SolutionTwoPointers().findRightInterval(iv),
    "submit_heap": lambda iv: SolutionHeap().findRightInterval(iv),
}


def check_all(intervals, expected, label):
    for key, fn in impls.items():
        actual = fn(intervals)
        assert actual == expected, (label, key, intervals, actual, expected)
    print("PASS", label)


for idx, (iv, expected) in enumerate(examples, 1):
    check_all(iv, expected, f"ví dụ {idx}")

PASS ví dụ 1
PASS ví dụ 2
PASS ví dụ 3


### 12.1. Các trường hợp biên

In [22]:
check_all([[5, 5]], [0], "một đoạn có start == end: right interval là chính nó")
check_all([[1, 2]], [-1], "một đoạn, start < end")
check_all([[1, 1], [2, 2]], [0, 1], "hai đoạn độ dài 0")
check_all([[1, 3], [2, 4], [3, 5]], [2, -1, -1], "start == end của đoạn khác")
check_all([[1, 10], [2, 3], [4, 5], [6, 7]], [-1, 2, 3, -1], "dây chuyền right interval")
check_all([[-3, -1], [-1, 2], [2, 6]], [1, 2, -1], "có toạ độ âm, chạm đầu mút")
check_all([[-1000000, 1000000], [1000000, 1000000]], [1, 1], "đúng biên toạ độ")
check_all([[1, 5], [2, 5], [3, 5], [4, 5]], [-1, -1, -1, -1], "mọi end đều lớn hơn mọi start")
check_all([[0, 0], [1, 1], [2, 2], [3, 3]], [0, 1, 2, 3], "mọi đoạn trỏ về chính mình")

PASS một đoạn có start == end: right interval là chính nó
PASS một đoạn, start < end
PASS hai đoạn độ dài 0
PASS start == end của đoạn khác
PASS dây chuyền right interval
PASS có toạ độ âm, chạm đầu mút
PASS đúng biên toạ độ
PASS mọi end đều lớn hơn mọi start
PASS mọi đoạn trỏ về chính mình


### 12.2. Phản ví dụ cho các cách làm sai

Hai lỗi phổ biến: (a) dùng `bisect_right` (bỏ sót `start == end`); (b) sắp xếp `intervals` rồi dùng chỉ số **sau khi sắp** làm đáp án.

In [23]:
def wrong_bisect_right(intervals):
    n = len(intervals)
    arr = sorted((st, i) for i, (st, _) in enumerate(intervals))
    starts = [st for st, _ in arr]
    ans = [-1] * n
    for i, (_, ed) in enumerate(intervals):
        k = bisect.bisect_right(starts, ed)      # SAI: bỏ sót start == ed
        if k < n:
            ans[i] = arr[k][1]
    return ans


def wrong_lost_index(intervals):
    arr = sorted(intervals)                      # SAI: sau khi sắp mất chỉ số gốc
    starts = [st for st, _ in arr]
    ans = []
    for _, ed in arr:
        k = bisect.bisect_left(starts, ed)
        ans.append(k if k < len(arr) else -1)    # k là chỉ số trong mảng đã sắp, không phải mảng gốc
    return ans


print("bisect_right:", wrong_bisect_right([[3, 4], [2, 3], [1, 2]]), "| đúng:", Solution().findRightInterval([[3, 4], [2, 3], [1, 2]]))
print("mất chỉ số :", wrong_lost_index([[3, 4], [2, 3], [1, 2]]), "| đúng:", Solution().findRightInterval([[3, 4], [2, 3], [1, 2]]))
assert wrong_bisect_right([[3, 4], [2, 3], [1, 2]]) != [-1, 0, 1]
assert wrong_lost_index([[3, 4], [2, 3], [1, 2]]) != [-1, 0, 1]

bisect_right: [-1, -1, 0] | đúng: [-1, 0, 1]
mất chỉ số : [1, 2, -1] | đúng: [-1, 0, 1]


### 12.3. Đối chiếu ngẫu nhiên (có seed)

Sinh `intervals` đúng ràng buộc: `start` **đôi một khác nhau**, `start <= end`, toạ độ trong `[-10^6, 10^6]`. Ta so mọi cách với vét cạn trên nhiều kích thước, cả miền toạ độ rất hẹp (nhiều va chạm `start == end`) lẫn miền rộng. Thuộc tính thêm: nếu `ans[i] = j != -1` thì `start_j >= end_i`, và **không có** `start` nào nằm giữa `[end_i, start_j)`.

In [24]:
rng = random.Random(436)


def gen(n, lo, hi):
    starts = rng.sample(range(lo, hi + 1), n)            # start đôi một khác nhau
    return [[s, rng.randint(s, hi)] for s in starts]


for trial in range(1500):
    n = rng.randint(1, 12)
    iv = gen(n, -8, 8)
    ref = find_right_bruteforce(iv)
    for key, fn in impls.items():
        assert fn(iv) == ref, (key, iv)
print("PASS 1500 input nhỏ (toạ độ hẹp, nhiều va chạm)")

for trial in range(300):
    n = rng.randint(50, 400)
    iv = gen(n, -10**6, 10**6) if trial % 2 else gen(n, -500, 500)
    ref = find_right_bruteforce(iv)
    for key, fn in impls.items():
        assert fn(iv) == ref, (key, n)
print("PASS 300 input vừa")

for trial in range(300):
    iv = gen(rng.randint(1, 40), -30, 30)
    ans = Solution().findRightInterval(iv)
    starts = sorted(s for s, _ in iv)
    for i, j in enumerate(ans):
        end_i = iv[i][1]
        if j == -1:
            assert all(s < end_i for s in starts)
        else:
            assert iv[j][0] >= end_i
            assert not any(end_i <= s < iv[j][0] for s in starts)
print("PASS thuộc tính: ans[i] đúng là start nhỏ nhất >= end_i")

big = gen(20000, -10**6, 10**6)
res = Solution().findRightInterval(big)
print("n = 2*10^4 ->", len(res), "phần tử, số -1:", res.count(-1))

PASS 1500 input nhỏ (toạ độ hẹp, nhiều va chạm)


PASS 300 input vừa
PASS thuộc tính: ans[i] đúng là start nhỏ nhất >= end_i
n = 2*10^4 -> 20000 phần tử, số -1: 3


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Sắp xếp xong làm mất chỉ số gốc

In [25]:
def sai_mat_chi_so(intervals):
    n = len(intervals)
    sorted_iv = sorted(intervals)                     # sắp theo start
    starts = [s for s, _ in sorted_iv]
    ans = [-1] * n
    for i, (_, ed) in enumerate(intervals):
        k = bisect.bisect_left(starts, ed)
        if k < n:
            ans[i] = k                                # SAI: k là vị trí trong mảng đã sắp
    return ans


iv = [[3, 4], [2, 3], [1, 2]]
print("sai:", sai_mat_chi_so(iv), "| đúng:", Solution().findRightInterval(iv))

sai: [-1, 2, 1] | đúng: [-1, 0, 1]


**Vì sao sai:** đề hỏi chỉ số trong mảng **ban đầu**. Sau khi sắp, `k` là vị trí trong mảng mới. **Cách sửa:** kèm chỉ số gốc vào từng phần tử trước khi sắp: `sorted((start, i) ...)` rồi trả `arr[k][1]`.

### Lỗi 2: Dùng `bisect_right` hoặc điều kiện `>`

In [26]:
def sai_bisect_right(intervals):
    n = len(intervals)
    arr = sorted((st, i) for i, (st, _) in enumerate(intervals))
    starts = [st for st, _ in arr]
    ans = [-1] * n
    for i, (_, ed) in enumerate(intervals):
        k = bisect.bisect_right(starts, ed)           # SAI
        if k < n:
            ans[i] = arr[k][1]
    return ans


iv = [[1, 2], [2, 3], [3, 4]]
print("sai:", sai_bisect_right(iv), "| đúng:", Solution().findRightInterval(iv))

sai: [2, -1, -1] | đúng: [1, 2, -1]


**Vì sao sai:** đề cho phép `start_j == end_i` (`>=`). `bisect_right` bỏ qua phần tử bằng `end_i`, nhảy sang phần tử lớn hơn hẳn. **Cách sửa:** `bisect_left`.

### Lỗi 3: Quên trường hợp `k == n` (đi quá cuối)

In [27]:
def sai_quen_het_mang(intervals):
    n = len(intervals)
    arr = sorted((st, i) for i, (st, _) in enumerate(intervals))
    starts = [st for st, _ in arr]
    ans = []
    for i, (_, ed) in enumerate(intervals):
        k = bisect.bisect_left(starts, ed)
        ans.append(arr[k][1])                         # SAI: k có thể bằng n -> IndexError
    return ans


try:
    sai_quen_het_mang([[1, 2]])
except IndexError as e:
    print("IndexError:", e)
print("đúng:", Solution().findRightInterval([[1, 2]]))

IndexError: list index out of range
đúng: [-1]


**Vì sao sai:** nếu không có start nào `>= end_i` thì `bisect_left` trả `len(starts)`, truy cập `arr[n]` gây `IndexError`. **Cách sửa:** luôn kiểm tra `k < n` rồi mới đọc, ngược lại ghi `-1`.

### Lỗi 4: Loại trừ chính đoạn `i` khỏi ứng viên

In [28]:
def sai_bo_chinh_no(intervals):
    n = len(intervals)
    ans = []
    for i in range(n):
        best = None
        for j in range(n):
            if j == i:
                continue                              # SAI: đề cho phép j == i
            s = intervals[j][0]
            if s >= intervals[i][1] and (best is None or s < intervals[best][0]):
                best = j
        ans.append(-1 if best is None else best)
    return ans


iv = [[5, 5]]
print("sai:", sai_bo_chinh_no(iv), "| đúng:", Solution().findRightInterval(iv))

sai: [-1] | đúng: [0]


**Vì sao sai:** đề viết rõ "`i` may equal `j`". Đoạn có `start == end` (độ dài `0`) chính là right interval của bản thân. **Cách sửa:** không loại `j == i`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "**Với mỗi phần tử, tìm phần tử nhỏ nhất `>= x` / lớn nhất `<= x` trong một tập**" ⇒ sắp xếp rồi `bisect_left` / `bisect_right - 1`.
- Cần trả về **chỉ số gốc** ⇒ luôn đóng gói `(giá_trị, chỉ_số)` trước khi sắp.
- `bisect_left(a, x)`: có phần tử `== x` thì dừng **ở** nó (`>= x`); `bisect_right(a, x)`: dừng **sau** nó (`> x`).
- Nếu cả danh sách câu hỏi cũng sắp được ⇒ hai con trỏ / heap tránh tìm nhị phân, hay gặp ở lịch trình, sự kiện.
- Khi đề nói "giá trị duy nhất", dùng `dict` giá trị → chỉ số là cách gọn nhất để tra ngược.

## 15. Biến thể của bài

1. **Left interval:** `end_j <= start_i` và `end_j` lớn nhất (đối xứng, dùng `bisect_right - 1` trên các `end`).
2. **Đoạn bắt đầu muộn nhất nhưng vẫn `< end_i`** (cận trên chặt).
3. **Start không duy nhất:** cần quy tắc phá hoà (ví dụ lấy chỉ số nhỏ nhất) — sắp theo `(start, index)` vẫn dùng được.
4. **Find Right Interval chuỗi:** đi theo `ans` liên tiếp `i -> ans[i] -> ans[ans[i]]` tới khi `-1`, đếm độ dài dây chuyền dài nhất.
5. Bài liên quan: **Non-overlapping Intervals (435)**, **Merge Intervals (56)**, **Insert Interval (57)**, **Next Greater Element**, **Minimum Interval to Include Each Query (1851)**.

In [29]:
def find_left_interval(intervals):
    """Biến thể 1: left interval = đoạn j có end_j <= start_i và end_j lớn nhất (chỉ số, hoặc -1)."""
    n = len(intervals)
    arr = sorted((ed, i) for i, (_, ed) in enumerate(intervals))
    ends = [e for e, _ in arr]
    ans = [-1] * n
    for i, (st, _) in enumerate(intervals):
        k = bisect.bisect_right(ends, st) - 1
        if k >= 0:
            ans[i] = arr[k][1]
    return ans


def longest_chain(intervals):
    """Biến thể 4: độ dài dây chuyền right interval dài nhất (đếm số đoạn, bỏ vòng j == i)."""
    nxt = Solution().findRightInterval(intervals)
    best = 0
    for i in range(len(intervals)):
        length, cur, seen = 1, i, {i}
        while nxt[cur] != -1 and nxt[cur] not in seen:
            cur = nxt[cur]
            seen.add(cur)
            length += 1
        best = max(best, length)
    return best


print(find_left_interval([[3, 4], [2, 3], [1, 2]]))        # [1, 2, -1]
print(longest_chain([[3, 4], [2, 3], [1, 2]]))              # 3 : [1,2] -> [2,3] -> [3,4]
assert find_left_interval([[3, 4], [2, 3], [1, 2]]) == [1, 2, -1]
assert longest_chain([[3, 4], [2, 3], [1, 2]]) == 3

[1, 2, -1]
3


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Quét toàn bộ | với mỗi `i`, duyệt mọi `j` | `O(n²)` | `O(1)` | oracle kiểm tra, `n` nhỏ |
| 2. Sắp `(start, i)` + `bisect_left` | lower bound của `end_i` | `O(n log n)` | `O(n)` | **nên nộp**, gọn nhất |
| 3. Hai con trỏ | sắp cả `start`, `end`; con trỏ chỉ đi tới | `O(n log n)` | `O(n)` | khi câu hỏi đã sắp sẵn |
| 4. Min-heap | loại dần các start quá nhỏ | `O(n log n)` | `O(n)` | xử lý offline, dữ liệu theo dòng |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `bisect.bisect_left([2,4,6], 4)`, `bisect.bisect_right([2,4,6], 4)`, `bisect.bisect_left([2,4,6], 7)`, `Solution().findRightInterval([[2,2]])`, `Solution().findRightInterval([[1,3],[3,5],[5,7]])`.

**Bài 2 (tính tay).** Với `intervals = [[4,6],[1,3],[7,9],[3,4]]`: lập bảng `i`, `start`, `end`, các start `>= end`, đáp án.

**Bài 3 (điền TODO).** Hoàn thành `find_right_todo(intervals)` theo Cách 2.

In [30]:
def find_right_todo(intervals):
    # TODO 1: arr = sorted((start, index)) từ intervals (dùng enumerate)
    # TODO 2: starts = list chỉ chứa các start của arr
    # TODO 3: với mỗi i, end: k = bisect.bisect_left(starts, end)
    # TODO 4: nếu k < n thì ans[i] = arr[k][1], ngược lại -1
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `find_right_dict(intervals)`: dùng `dict` `start -> chỉ số` và `bisect_left` trên danh sách `start` đã sắp (không dùng tuple `(start, index)`).

**Bài 5 (biến thể / mini challenge).** (a) Viết `count_with_right(intervals)` trả về **số đoạn có right interval** (số `ans[i] != -1`). (b) Viết `find_right_interval_dup(intervals)` cho phiên bản **start có thể trùng**: nếu nhiều đoạn có cùng start nhỏ nhất thoả điều kiện thì chọn đoạn có **chỉ số nhỏ nhất**.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `1`, `2`, `3`, `[0]`, `[1, 2, -1]`. Bài 3: `arr = sorted((st, i) for i, (st, _) in enumerate(intervals))`.
- **Mức 2 (vừa):** Bài 2: `[4,6]` cần start `>= 6` -> `7` (đoạn 2); `[1,3]` cần `>= 3` -> `3` (đoạn 3) vì start 3 < 4; `[7,9]` cần `>= 9` -> `-1`; `[3,4]` cần `>= 4` -> `4` (đoạn 0). Đáp án `[2, 3, -1, 0]`. Bài 4: `idx = {l: i ...}`; `starts = sorted(idx)`; `ans[i] = idx[starts[k]]`.
- **Mức 3 (sâu):** Bài 5(a): `sum(1 for j in ans if j != -1)`. 5(b): sắp theo `(start, index)` — với start trùng thì chỉ số nhỏ đứng trước; dùng `bisect_left` trên danh sách start thì `k` trỏ vào phần tử đầu của cụm start trùng, đó chính là chỉ số nhỏ nhất.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [31]:
# Bài 1
assert bisect.bisect_left([2, 4, 6], 4) == 1
assert bisect.bisect_right([2, 4, 6], 4) == 2
assert bisect.bisect_left([2, 4, 6], 7) == 3
assert Solution().findRightInterval([[2, 2]]) == [0]
assert Solution().findRightInterval([[1, 3], [3, 5], [5, 7]]) == [1, 2, -1]

# Bài 2
assert Solution().findRightInterval([[4, 6], [1, 3], [7, 9], [3, 4]]) == [2, 3, -1, 0]


# Bài 3
def find_right_todo_ans(intervals):
    n = len(intervals)
    arr = sorted((st, i) for i, (st, _) in enumerate(intervals))
    starts = [st for st, _ in arr]
    ans = [-1] * n
    for i, (_, ed) in enumerate(intervals):
        k = bisect.bisect_left(starts, ed)
        if k < n:
            ans[i] = arr[k][1]
    return ans


for iv, expected in examples:
    assert find_right_todo_ans(iv) == expected


# Bài 4
def find_right_dict(intervals):
    idx = {l: i for i, (l, _) in enumerate(intervals)}
    starts = sorted(idx)
    ans = []
    for _, ed in intervals:
        k = bisect.bisect_left(starts, ed)
        ans.append(idx[starts[k]] if k < len(starts) else -1)
    return ans


for iv, expected in examples:
    assert find_right_dict(iv) == expected


# Bài 5
def count_with_right(intervals):
    return sum(1 for j in Solution().findRightInterval(intervals) if j != -1)


def find_right_interval_dup(intervals):
    n = len(intervals)
    arr = sorted((st, i) for i, (st, _) in enumerate(intervals))
    starts = [st for st, _ in arr]
    ans = [-1] * n
    for i, (_, ed) in enumerate(intervals):
        k = bisect.bisect_left(starts, ed)
        if k < n:
            ans[i] = arr[k][1]
    return ans


assert count_with_right([[3, 4], [2, 3], [1, 2]]) == 2
assert count_with_right([[1, 2]]) == 0
assert find_right_interval_dup([[5, 6], [2, 3], [2, 4], [3, 3]]) == [-1, 3, 0, 3]
print("Tất cả đáp án bài tập đều đúng.")

Tất cả đáp án bài tập đều đúng.


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (với mỗi đoạn, tìm start nhỏ nhất `>= end`)

```python
class Solution:
    def findRightInterval(self, intervals: List[List[int]]) -> List[int]:
        n = len(intervals)
        ans = [-1] * n
        arr = sorted((st, i) for i, (st, _) in enumerate(intervals))   # (start, chỉ số gốc)
        starts = [st for st, _ in arr]
        for i, (_, ed) in enumerate(intervals):
            j = bisect.bisect_left(starts, ed)                          # start nhỏ nhất >= ed
            if j < n:
                ans[i] = arr[j][1]
        return ans
```

**Checklist**

- [ ] Đã đóng gói `(start, index)` trước khi sắp để giữ chỉ số gốc?
- [ ] Dùng `bisect_left` (điều kiện `>=`), không phải `bisect_right`?
- [ ] Đã kiểm tra `j < n` trước khi đọc `arr[j]` và trả `-1` khi vượt cuối?
- [ ] Không loại trừ `j == i` (đoạn `start == end` trỏ về chính nó)?
- [ ] Đã thử `[[1,2]]` (`[-1]`), `[[3,4],[2,3],[1,2]]` (`[-1,0,1]`), `[[1,4],[2,3],[3,4]]` (`[-1,2,-1]`) và `[[5,5]]` (`[0]`)?
- [ ] Skeleton đúng `class Solution` / `def findRightInterval(self, intervals: List[List[int]]) -> List[int]`?